# Pipeline Data Analysis

Some text about what is here and how to use it


## 1. Setup


In [ ]:
import os
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import pygwalker as pyg
import seaborn as sns
from ipyfilechooser import FileChooser
from IPython.display import Image, display
from itables import init_notebook_mode

from spot_detector.config import load_config

init_notebook_mode(all_interactive=True, connected=False)

In [ ]:
project_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/config.yml").exists()
)
os.chdir(project_root)

In [ ]:
config_path = Path("configs/config.yml")
config = load_config(config_path)

mode = "3D" if config.mode.do_3d else "2D"
out_dir = Path(config.paths.out_dir)
tab_dir = out_dir / "tables"

## 2. Select Table

with gui chose table to analyse


In [ ]:
fc = FileChooser(str(tab_dir), filter_pattern=["*.csv"], select_default=False)
display(fc)

In [ ]:
csv_path = Path(fc.selected) if fc.selected else tab_dir / f"_run_objects_{mode}.csv"
df = pd.read_csv(csv_path, keep_default_na=False, na_values=[""]).dropna(
    how="all", axis=1
)
print(
    f"Loaded {csv_path.name}  —  {len(df)} rows, {df['Condition'].nunique()} condition(s)"
)

df

## 3. Exploratory Data Analysis

looking at data, tables, basic stats etc

maybe something for exploratory tables? stats?

pygwalker for interactive exploratory plotting


In [ ]:
# add factor colums (optional)
# One row of metadata per Condition. Add as many columns as you need
# (concentration, drug name, genotype, ...) - TODO: fill in your real conditions.

ADD_CONDITION_META = (
    False  # flip to True once condition_meta below has your real conditions
)

if ADD_CONDITION_META:
    # One row of metadata per Condition. Add as many columns as you need
    # (concentration, drug name, genotype, ...) - TODO: fill in your real conditions.
    condition_meta = {
        "Control": {"Drug_Conc_uM": 0},
        "DrugA_10uM": {"Drug_Conc_uM": 10},
        "DrugA_50uM": {"Drug_Conc_uM": 50},
        # ...
    }
    meta_df = pd.DataFrame.from_dict(condition_meta, orient="index")
    df = df.drop(
        columns=[c for c in meta_df.columns if c in df.columns], errors="ignore"
    ).merge(meta_df, left_on="Condition", right_index=True, how="left")

    missing = set(df["Condition"]) - set(condition_meta)
    if missing:
        print(f"WARNING: no metadata mapping for condition(s): {missing}")

In [ ]:
# some summary stats of the table
df.describe().T

In [ ]:
# --- 3.3 Stats per group ---
group_dd = widgets.Dropdown(
    options=[c for c in df.columns if df[c].nunique() <= 15], description="Group by:"
)
metric_dd = widgets.Dropdown(
    options=df.select_dtypes(include="number").columns.tolist(), description="Metric:"
)
stats_view = widgets.Output()


def update_stats(*_):
    stats_view.clear_output(wait=True)
    with stats_view:
        display(df.groupby(group_dd.value)[metric_dd.value].describe())  # type: ignore


group_dd.observe(update_stats, names="value")
metric_dd.observe(update_stats, names="value")
update_stats()
display(widgets.HBox([group_dd, metric_dd]), stats_view)

In [ ]:
# look at the interactive packages
walker = pyg.walk(df)

## 4. Browse Individual Outputs

same as in `pipeline.run.ipynb`, to see default plots and even qc figs to see if stuff makes sense


In [ ]:
fc = FileChooser(
    str(out_dir / "figures"), filter_pattern=["*.png"], select_default=False
)
view = widgets.Output()


def on_pick(chooser):
    if chooser.selected is None:
        return
    p = Path(chooser.selected)
    view.clear_output(wait=True)
    with view:
        display(Image(str(p), width=1000))


fc.register_callback(on_pick)
display(fc, view)

## 5. Final Figure Plot

place to construct figure for presentations and reports


In [ ]:
def plot_metric_by_group(
    df: pd.DataFrame,
    metric: str,
    group: str,
    hue: str | None = None,
    scale: str = "log",  # "log" | "linear"
    category_axis: str = "auto",  # "auto" | "x" | "y"
    order: list | None = None,
    show_points: bool = True,
    label_len_threshold: int = 12,
    log_offset: float
    | None = None,  # e.g. 1 -- shift `metric` before log-scaling so zeros/negatives stay visible
    title: str | None = None,
    save_as: Path | None = None,
    figsize: tuple = (9, 5),
):
    """Boxplot (+ optional stripplot) of `metric` split by `group`.

    category_axis="auto": categories go on x (vertical boxes) unless the
    longest label exceeds `label_len_threshold` chars, then flips to y.
    scale="log" on data with non-positive values normally makes matplotlib
    silently drop those points off the axis. Pass log_offset (e.g. 1e-8) to
    shift the plotted values instead - loudly, via a printed note, and only
    for this plot (the underlying `df` is never modified).
    """
    labels = order if order is not None else df[group].astype(str).unique().tolist()
    axis = (
        ("y" if max(len(str(lab)) for lab in labels) > label_len_threshold else "x")
        if category_axis == "auto"
        else category_axis
    )

    plot_df = df
    if scale == "log" and (df[metric] <= 0).any():
        if log_offset is not None:
            plot_df = df.copy()
            plot_df[metric] = plot_df[metric] + log_offset
            print(
                f"NOTE: '{metric}' has non-positive values - added {log_offset:g} before "
                "log-scaling so they stay visible (df itself is unchanged)."
            )
            if (plot_df[metric] <= 0).any():
                print(
                    f"WARNING: log_offset={log_offset:g} wasn't enough - '{metric}' still has "
                    "non-positive values after the shift; try a larger offset."
                )
        else:
            print(
                f"WARNING: '{metric}' has non-positive values - they'll be dropped from the "
                "log-scaled axis (boxes/whiskers may look clipped). Pass log_offset=1 (or "
                "similar) to shift instead, or scale='linear' to avoid the issue entirely."
            )

    fig, ax = plt.subplots(figsize=figsize)
    xy = dict(x=group, y=metric) if axis == "x" else dict(x=metric, y=group)

    sns.boxplot(
        data=plot_df, hue=hue, order=order, ax=ax, palette="Set2" if hue else None, **xy
    )
    if show_points:
        sns.stripplot(
            data=plot_df,
            hue=hue,
            order=order,
            ax=ax,
            dodge=bool(hue),
            size=5,
            alpha=0.7,
            edgecolor="white",
            linewidth=0.5,
            palette="Set2" if hue else None,
            color=None if hue else "0.25",
            **xy,
        )
        if hue:
            # box + strip each add one legend entry per hue level -> dedup
            handles, leg_labels = ax.get_legend_handles_labels()
            n_levels = df[hue].nunique()
            ax.legend(handles[:n_levels], leg_labels[:n_levels], title=hue)

    (ax.set_yscale if axis == "x" else ax.set_xscale)(scale)

    ax.set_title(title or f"{metric} by {group}")
    plt.tight_layout()
    if save_as:
        save_as.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_as, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# --- knobs: change these; leave the plotting code below alone ---
METRIC = "Spot_Count"
GROUP = "Condition"
TITLE = "Spot Count Distribution by Condition"
SCALE = "log"
LOG_OFFSET = 1
SAVE_AS = None  # e.g. Path("output/figures/lab_meeting/spot_count_by_condition.png")

In [ ]:
plot_metric_by_group(
    df, metric=METRIC, group=GROUP, scale=SCALE, log_offset=LOG_OFFSET, save_as=SAVE_AS
)